In [35]:
import re
import collections
import torch
from d2l import torch as d2l


# ============================================================
# 1. 下载并读取《时间机器》
# ============================================================

d2l.DATA_HUB["time_machine"] = (
    d2l.DATA_URL + "timemachine.txt",
    "090b5e7e70c295757f55df93cb0a180b9691891a"
)


def read_time_machine() -> list[str]:
    file_path = d2l.download(
        "time_machine"
    )

    with open(
        file_path,
        "r"
    ) as file:
        lines = file.readlines()
    

    # 删除标点、转小写、删除首尾空格
    lines = [
        re.sub(
            "[^A-Za-z]+",
            " ",
            line
        ).strip().lower()

        for line in lines
    ]
    
    return lines


# ============================================================
# 2. 将每行拆成单词或字符
# ============================================================

def tokenize(
    lines: list[str],
    token: str = "word"
) -> list[list[str]]:

    if token == "word":
        return [
            line.split()
            for line in lines
        ]

    if token == "char":
        return [
            list(line)
            for line in lines
        ]

    raise ValueError(
        f"未知词元类型：{token}"
    )


# ============================================================
# 3. 统计每个词元出现的次数
# ============================================================

def count_corpus(tokens):
    # 如果tokens是二维列表，就展平成一维
    if (
        len(tokens) == 0
        or isinstance(tokens[0], list)
    ):
        tokens = [
            token
            for line in tokens
            for token in line
        ]

    return collections.Counter(tokens)


# ============================================================
# 4. 建立词表
# ============================================================

class Vocab:
    def __init__(
        self,
        tokens=None,
        min_freq: int = 0,
        reserved_tokens=None
    ):
        if tokens is None:
            tokens = []

        if reserved_tokens is None:
            reserved_tokens = []

        # 统计词频并从高到低排序
        counter = count_corpus(tokens)

        self.token_freqs = sorted(
            counter.items(),
            key=lambda item: item[1],
            reverse=True
        )

        # 下标0固定分配给未知词元
        self.idx_to_token = [
            "<unk>"
        ] + reserved_tokens

        self.token_to_idx = {
            token: index
            for index, token
            in enumerate(self.idx_to_token)
        }

        # 为其余词元分配编号
        for token, frequency in self.token_freqs:
            if frequency < min_freq:
                break

            if token not in self.token_to_idx:
                self.idx_to_token.append(
                    token
                )

                self.token_to_idx[token] = (
                    len(self.idx_to_token) - 1
                )

    def __len__(self):
        return len(
            self.idx_to_token
        )

    def __getitem__(self, tokens):
        # 输入单个词元
        if not isinstance(
            tokens,
            (list, tuple)
        ):
            return self.token_to_idx.get(
                tokens,
                self.unk
            )

        # 输入词元列表
        return [
            self[token]
            for token in tokens
        ]

    def to_tokens(self, indices):
        # 输入单个编号
        if not isinstance(
            indices,
            (list, tuple)
        ):
            return self.idx_to_token[
                indices
            ]

        # 输入编号列表
        return [
            self.idx_to_token[index]
            for index in indices
        ]

    @property
    def unk(self):
        return 0


# ============================================================
# 5. 整合：生成数字语料库和词表
# ============================================================

def load_corpus_time_machine(
    max_tokens: int = -1
):
    # 读取文本
    lines = read_time_machine()

    # 按字符拆分
    tokens = tokenize(
        lines,
        token="char"
    )

    # 建立字符词表
    vocab = Vocab(tokens)

    # 将所有字符转换成数字，并展平成一维列表
    corpus = [
        vocab[token]
        for line in tokens
        for token in line
    ]

    # 限制最大字符数量
    if max_tokens > 0:
        corpus = corpus[
            :max_tokens
        ]

    return corpus, vocab


# ============================================================
# 6. 查看结果
# ============================================================

corpus, vocab = (
    load_corpus_time_machine()
)

print("语料长度：", len(corpus))
print("词表大小：", len(vocab))

print("前20个数字：")
print(corpus[:20])

print("还原成字符：")
print(
    vocab.to_tokens(
        corpus[:20]
    )
)

语料长度： 170580
词表大小： 28
前20个数字：
[3, 9, 2, 1, 3, 5, 13, 2, 1, 13, 4, 15, 9, 5, 6, 2, 1, 21, 19, 1]
还原成字符：
['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm', 'a', 'c', 'h', 'i', 'n', 'e', ' ', 'b', 'y', ' ']


In [36]:
def my_corpus_vocab_maker(max_token=None):
    with open("../data/timemachine.txt",'r') as file:
        text=file.read().lower()
    #print(text)
    text=re.sub("[^A-Za-z]+",' ',text).strip().lower()
    #print(text)
    tokens=list(text)
    idx_to_token=[*(['<unk>']+list(set(tokens)))]
    token_to_idx={
        token:idx
        for idx,token in enumerate(idx_to_token)
    }
    corpus=torch.tensor([token_to_idx[token] for token in tokens])
    vocab={
        "idx_to_token":idx_to_token,
        "token_to_idx":token_to_idx

    }
    return corpus,vocab

In [37]:
corpus,vocab=my_corpus_vocab_maker()
print(corpus[:20])
print(
    [vocab["idx_to_token"][idx] for idx in corpus[:20].tolist()]
)

tensor([12, 15, 21,  1, 12, 23, 22, 21,  1, 22,  3, 24, 15, 23,  9, 21,  1, 13,
        26,  1])
['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm', 'a', 'c', 'h', 'i', 'n', 'e', ' ', 'b', 'y', ' ']
